In [ ]:
import pandas as pd, numpy as np, re
import plotly.graph_objects as go

# Load
csv_path='data/Assignment4-data.csv'
df=pd.read_csv(csv_path)

# Order age groups (young -> old)
age_order=sorted(df['Age group'].unique(), key=lambda s: int(re.match(r'\s*(\d+)', str(s)).group(1)), reverse=True)

# Cohorts: oldest -> youngest
cohort_cols = [c for c in df.columns if c != "Age group"]
cohort_order = sorted(cohort_cols, key=lambda s: int(re.match(r"\s*(\d{4})", s).group(1)))

# Z matrix
z = np.array([df.loc[df["Age group"] == ag, cohort_order].iloc[0].to_list() for ag in age_order],
             dtype=float)

# Missing overlay + main heatmap values
z_main    = np.where(np.isnan(z), None, z)

# ---- DISCRETE BINS ----
# <10, 10–<20, 20–<25, 25–<30, 30–<35, ≥35  (cap at 40 for color normalization)
zmin, zmax = 0.0, 40.0
pos = lambda v: v / zmax


bounds = [0, 10, 20, 25, 30, 35, 40]
colors = ['#006837', '#66bd63', '#a6d96a', '#fee08b', '#fdae61', '#d73027']  # good->bad

# Stepwise colorscale (MUST be non-decreasing in position)
colorscale = []
for (lo, hi), col in zip(zip(bounds[:-1], bounds[1:]), colors):
    colorscale.append((pos(lo), col))
    colorscale.append((pos(hi), col))

# Colorbar ticks at bin midpoints
mid = [(bounds[i] + bounds[i+1]) / 2 for i in range(len(bounds) - 1)]
ticktext = ['<10%', '10–20%', '20–25%', '25–30%', '30–35%', '≥35%']

fig = go.Figure()


# Main heatmap
fig.add_trace(go.Heatmap(
    z=z_main,
    x=cohort_order,
    y=age_order,
    zmin=zmin, zmax=zmax,
    colorscale=colorscale,
    colorbar=dict(title="Obesity prevalence (%)",
                  tickmode="array",
                  tickvals=mid,
                  ticktext=ticktext),
    hovertemplate="Age group: %{y}<br>Cohort: %{x}<br>Obesity: %{z:.2f}%<extra></extra>"
))

# Youngest at bottom, oldest at top:
fig.update_yaxes(categoryorder='array', categoryarray=age_order, autorange='reversed')

fig.update_layout(
    title="Obesity prevalence (%) by age group and birth cohort",
    xaxis_title="Birth cohort",
    yaxis_title="Age group",
    template="plotly_white")


annotation=(
    'Green→red scale encodes obesity prevalence (BMI ≥ 30). '
    'Bins follow CDC obesity-prevalence map legend: <20% (lower/better), 20–<25, 25–<30, 30–<35, ≥35% (higher/worse). '
    'Missing/unavailable cohort–age cells are shown as light transparent gray.'
)
fig.add_annotation(text=annotation, xref='paper', yref='paper', x=0, y=-0.20,
                   showarrow=False, align='left', font=dict(size=11))

html_path='obesity_heatmap_medical_palette_missinggray.html'
fig.write_html(html_path, include_plotlyjs='cdn')

png_path='obesity_heatmap_medical_palette_missinggray.png'
fig.write_image(png_path, width=1200, height=680, scale=2)

svg_path='obesity_heatmap_medical_palette_missinggray.svg'
fig.write_image(svg_path, format="svg")

html_path, png_path, svg_path


('obesity_heatmap_medical_palette_missinggray.html',
 'obesity_heatmap_medical_palette_missinggray.png',
 'obesity_heatmap_medical_palette_missinggray.svg')